# Iris dataset

O conjunto de dados flor Iris ou conjunto de dados Iris de Fisher é um conjunto de dados multivariados introduzido pelo estatístico e biólogo britânico Ronald Fisher em seu artigo de 1936, O uso de múltiplas medições em problemas taxonômicos, como um exemplo de análise discriminante linear.

Fonte: [Wikipédia](https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://pt.wikipedia.org/wiki/Conjunto_de_dados_flor_Iris&ved=2ahUKEwiS696899KUAxVAqJUCHY75L7gQmhN6BAgWEAU&usg=AOvVaw1kjl6kG1h3QhXFTEY4y3w2)

# Load Iris Dataset

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

# Load the Iris dataset
iris = load_iris()

# Create a DataFrame from the data
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

# Add the target variable (species)
iris_df['species'] = iris.target

# Map target integers to species names
iris_df['species'] = iris_df['species'].map({i: name for i, name in enumerate(iris.target_names)})

# Display the first 5 rows of the DataFrame
display(iris_df.head())

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


### Dividir o Dataset em Treino e Teste

Agora vamos dividir o `iris_df` em conjuntos de treino e teste. O requisito é que o conjunto de teste contenha 5 instâncias de cada classe. Usaremos a função `train_test_split` da biblioteca `sklearn.model_selection` com o parâmetro `stratify` para garantir que a proporção das classes seja mantida tanto no conjunto de treino quanto no de teste.

In [ ]:
from sklearn.model_selection import train_test_split

# Separar as features (X) e o target (y)
X = iris_df.drop('species', axis=1)
y = iris_df['species']

# Definir o tamanho do conjunto de teste
# Temos 3 classes, e queremos 5 instâncias por classe no teste, totalizando 15 instâncias.
test_size_absolute = 15

# Dividir o dataset em treino e teste de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size_absolute, stratify=y, random_state=42
)

print(f"Shape do conjunto de treino (features): {X_train.shape}")
print(f"Shape do conjunto de treino (target): {y_train.shape}")
print(f"Shape do conjunto de teste (features): {X_test.shape}")
print(f"Shape do conjunto de teste (target): {y_test.shape}")

print("\nDistribuição das classes no conjunto de treino:")
display(y_train.value_counts())

print("\nDistribuição das classes no conjunto de teste:")
display(y_test.value_counts())

Shape do conjunto de treino (features): (135, 4)
Shape do conjunto de treino (target): (135,)
Shape do conjunto de teste (features): (15, 4)
Shape do conjunto de teste (target): (15,)

Distribuição das classes no conjunto de treino:


,count
species,
setosa,45
virginica,45
versicolor,45



Distribuição das classes no conjunto de teste:


,count
species,
versicolor,5
virginica,5
setosa,5


In [ ]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

### Calcular o Vetor Médio de Cada Classe

Vamos agora calcular o vetor médio (centroid) para cada uma das três classes (setosa, versicolor, virginica) usando os dados do conjunto de treino (`X_train` e `y_train`). O vetor médio representa o 'centro' de cada classe no espaço de características.

In [ ]:
import numpy as np

# Obter os nomes das classes
class_names = iris.target_names

# Dicionário para armazenar os vetores médios de cada classe
mean_vectors = {}

print("Vetores Médios para Cada Classe:")
for i, class_name in enumerate(class_names):
    # Filtrar X_train para obter apenas as instâncias desta classe
    # Como y_train contém strings dos nomes das espécies, podemos filtrar diretamente
    X_train_class = X_train[y_train == class_name]

    # Calcular o vetor médio para as features desta classe
    mean_vector = np.mean(X_train_class, axis=0)
    mean_vectors[class_name] = mean_vector

    print(f"  {class_name.capitalize()}: {mean_vector}")

# Opcional: Converter para DataFrame para melhor visualização
mean_vectors_df = pd.DataFrame(mean_vectors).T
mean_vectors_df.columns = iris.feature_names
print("\nVetores Médios (DataFrame):")
display(mean_vectors_df)

Vetores Médios para Cada Classe:
  Setosa: [5.01111111 3.43111111 1.47555556 0.25111111]
  Versicolor: [5.88666667 2.75333333 4.23555556 1.31777778]
  Virginica: [6.59111111 2.98666667 5.54222222 2.04666667]

Vetores Médios (DataFrame):


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
setosa,5.011111,3.431111,1.475556,0.251111
versicolor,5.886667,2.753333,4.235556,1.317778
virginica,6.591111,2.986667,5.542222,2.046667


### Classificação das Instâncias de Teste Usando Vetores Médios (Nearest Centroid Classifier)

Vamos classificar as instâncias do conjunto de teste (`X_test`) comparando cada uma com os vetores médios calculados. A classe de uma instância de teste será aquela cujo vetor médio for mais próximo (usando a distância Euclidiana).

Após a classificação, calcularemos a precisão total e a precisão para cada classe individualmente.

In [ ]:
from scipy.spatial.distance import euclidean

# Lista para armazenar as previsões
predictions = []

# Classificar cada instância do conjunto de teste
for i, test_instance in enumerate(X_test):
    min_distance = float('inf')
    predicted_class = None

    for class_name, mean_vec in mean_vectors.items():
        # Calcular a distância Euclidiana entre a instância de teste e o vetor médio da classe
        distance = euclidean(test_instance, mean_vec)

        if distance < min_distance:
            min_distance = distance
            predicted_class = class_name
    predictions.append(predicted_class)

# Converter as previsões para um array numpy para facilitar a comparação
predictions = np.array(predictions)

print("Previsões (primeiras 5):", predictions[:5])
print("Labels Reais (primeiras 5):", y_test[:5])

Previsões (primeiras 5): ['versicolor' 'virginica' 'virginica' 'versicolor' 'virginica']
Labels Reais (primeiras 5): [np.str_('versicolor') np.str_('virginica') np.str_('virginica')
 np.str_('versicolor') np.str_('virginica')]


### Avaliação do Classificador

Agora vamos calcular a acurácia total (porcentagem de acertos) e a acurácia para cada classe.

In [ ]:
# Calcular a acurácia total
correct_predictions_total = np.sum(predictions == y_test)
total_instances_test = len(y_test)
overall_accuracy = (correct_predictions_total / total_instances_test) * 100

print(f"Acurácia Total: {overall_accuracy:.2f}%")

print("\nAcurácia por Classe:")
# Calcular a acurácia para cada classe
for class_name in class_names:
    # Filtrar previsões e labels reais para esta classe
    class_indices = (y_test == class_name)
    true_labels_class = y_test[class_indices]
    predicted_labels_class = predictions[class_indices]

    correct_predictions_class = np.sum(predicted_labels_class == true_labels_class)
    total_instances_class = len(true_labels_class)

    if total_instances_class > 0:
        accuracy_class = (correct_predictions_class / total_instances_class) * 100
        print(f"  {class_name.capitalize()}: {accuracy_class:.2f}%")
    else:
        print(f"  {class_name.capitalize()}: Nenhuma instância desta classe no conjunto de teste.")

Acurácia Total: 93.33%

Acurácia por Classe:
  Setosa: 100.00%
  Versicolor: 80.00%
  Virginica: 100.00%
